<img src="logo.png" alt="Vegeta" width="240">

# Fidia — from a prompt to a 3D model

**Fidia** (Phidias, the sculptor of the Parthenon) turns a text prompt into a 3D model through a bounded loop:

`prompt → plan → generate CadQuery code → run it in a sandbox → mesh checks + five rendered views → review → revise`

- The model **writes code, never meshes**: a Dedalus `Design` whose `build()` returns named, coloured parts.
- The code runs **out of process** (a subprocess with a timeout, a memory cap and no API keys), never in this kernel.
- **Vegeta's checks decide validity** (valid solids, watertight, winding, triangle budget, export round-trip);
  the model's review of the rendered views is advice that scores and steers the next revision.
- Every revision is kept on disk, the **best valid one** is copied to `best/`, and the model is exported as
  **glTF/GLB and OBJ** (+ STL, STEP) and re-imported to prove the files work.
- Limits on iterations, minutes and tokens; `session.cancel()`, a `STOP` file or *Interrupt kernel* stop it cleanly.

With `ANTHROPIC_API_KEY` set, Claude plans, writes and reviews. Without a key, a scripted **demo agent** plays
the model (it always builds a stool), so every step below still runs. Setup and safety notes: `docs/fidia.md`.

In [ ]:
import json, os, shutil
from pathlib import Path
from IPython.display import display
from vegeta.fidia import DEMO_PROMPT, Limits, Session, agent_from_config, demo_agent

HAVE_KEY = bool(os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("ANTHROPIC_AUTH_TOKEN"))
PROMPT = "a small stool with three legs"          # with a key, try your own prompt
if HAVE_KEY:
    agent = agent_from_config("claude-opus-5", effort="high", review_effort="medium")
else:
    agent, PROMPT = demo_agent(), DEMO_PROMPT    # offline: scripted answers, no cost

RUN = Path("_runs/fidia/demo"); shutil.rmtree(RUN, ignore_errors=True)
session = Session(PROMPT, RUN, agent=agent, limits=Limits(max_iterations=5, max_minutes=15, max_tokens=300_000))
print("agent:", agent.describe()["provider"], "/", agent.describe()["model"])
print("run directory:", RUN)

## 1. Run the loop

One line per revision: build status, validity (Vegeta's checks), the review score, and whether it is the best so far.
It stops when a revision is **done** (valid, clean export, accepted with score ≥ 8, all acceptance checks met) or at
a limit.

In [ ]:
session.run();

## 2. The plan

The model's plan: overall size, named parts with colours, and the acceptance checks the reviewer answers.

In [ ]:
plan = session.plan
print(f"{plan['object']}: {plan['description']}\nsize (x, y, z mm): {plan['size_mm']}")
for p in plan["parts"]:
    print(f"  {p['name']:10s} {p['shape']}")
print("acceptance checks:", *[f"  - {a}" for a in plan["acceptance"]], sep="\n")

## 3. Every revision

The contact sheet is what the reviewer sees (front, right, top, two iso views and a legend). Checks are facts
measured by Vegeta; the review is the model's opinion.

In [ ]:
for rev in session.revisions:
    print(rev.text())
    display(rev.sheet(width=900))

In [ ]:
session.table()

## 4. The best valid revision — interactive preview

Rotate and zoom in a live notebook (a static image when run headless).

In [ ]:
best = session.best
print(best.text())
best.preview()

## 5. Export and re-import check

glTF/GLB are Y-up in metres (the glTF convention), OBJ/STL/STEP Z-up in millimetres like the CAD. Each file was read
back with two independent readers (trimesh and VTK) and compared with what was written: parts, triangles, bounds,
colours, and the sidecars (`model.bin` for glTF, `model.mtl` for OBJ).

In [ ]:
report = best.reimport
for fmt, entry in report["formats"].items():
    readers = {r: (entry[r].get("parts", entry[r].get("blocks")), entry[r]["triangles"]) for r in report["readers"] if r in entry}
    print(f"{fmt:5s} {entry['file']:11s} {'ok' if entry['ok'] else 'FAILED'}  (parts, triangles) by reader: {readers}")
print("\nfiles:", *[f"  {fmt:5s} {path}" for fmt, path in best.files().items()], sep="\n")

The same model back in Vegeta: read from the exported STEP (no generated code runs here) and measured by Dedalus.

In [ ]:
from vegeta.dedalus.geometry import Geometry

g = Geometry.from_step(best.files()["step"])
m = g.measure()
print(f"valid {m['valid']}, {m['n_solids']} solids, volume {m['volume'] / 1e3:.0f} cm³, "
      f"size {[round(d) for d in m['dimensions']]} mm")

## 6. Your feedback

Feedback goes to the next revision (`replan=True` also re-plans first). A revision that has seen your latest feedback
takes precedence for *best* over older ones.

In [ ]:
FEEDBACK = "make the seat blue and use four legs"     # change me
session.feedback(FEEDBACK)
session.run()
display(session.best.sheet(width=900))
session.table()

## 7. What is on disk

Everything above is in files: `run.json` (state, rewritten atomically), `events.jsonl` (append-only log),
`plan.json` (+ `plans/`), one `rev-NNN/` per revision (code, execution log, checks, renders, review, exports;
sealed by `revision.json`), and `best/` — a copy of the best valid revision. `Session.open(RUN, agent=...)` resumes
a run; `fidia resume` and `fidia show` do the same from a terminal.

In [ ]:
print(session.report())
for p in sorted(RUN.iterdir()):
    print(" ", p.name + ("/" if p.is_dir() else ""))
print("best/export:", sorted(f.name for f in (RUN / "best" / "export").iterdir()))

### Stopping, limits, costs

- `Limits(max_iterations, max_minutes, max_tokens, patience, accept_score)`; the token budget is checked before
  every model call.
- `session.cancel()` (from another thread or a button) abandons a running model call or build; creating a file
  `STOP` in the run directory or *Interrupt kernel* also stops it, and the state is saved.
- `Session(..., approve="ask")` asks before each generated file is run; `on_iteration=` can return feedback or `False`.
- One iteration with Claude is two calls (write code, review one image) plus a plan at the start; the report shows
  tokens and an estimated cost.